In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split,cross_val_score,StratifiedKFold,GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

#Load the cleaned.csv dataset
df=pd.read_csv("/content/cleaned_data.csv")
print(f"Cleaned data has been loaded Successfully with shape:{df.shape}")

y_reg = df["Item_Outlet_Sales"].copy()
y_clf = (y_reg > y_reg.median()).astype(int)

X = df.drop(columns=["Item_Outlet_Sales", "Item_Identifier"])

#Encode categorical column

cat_cols= X.select_dtypes(include=["object","string"]).columns.tolist()
print(f"Categorical Columns identified are:{cat_cols}")

# --- Ordinal encodings (natural order exists) ---

# Outlet_Size has a natural small < medium < high ordering
outlet_size_map={"Small":0,"Medium":1,"High":2}
X['Outlet_Size']=X['Outlet_Size'].map(outlet_size_map)

#Outlet_Location_Type: Tier 1 (largest/most developed) > Tier 2 > Tier 3 in the
# standard Indian retail classification, so tier number reflects an ordered
# decrease in market development. We map Tier 1 -> 2, Tier 2 -> 1, Tier 3 -> 0
# so that larger encoded values consistently correspond to "more developed".
outlet_tier_map = {"Tier 3": 0, "Tier 2": 1, "Tier 1": 2}
X["Outlet_Location_Type"] = X["Outlet_Location_Type"].map(outlet_tier_map)

# --- One-hot encodings (no natural order) ---
nominal_cols = ["Item_Fat_Content", "Item_Type", "Outlet_Identifier", "Outlet_Type"]
print(f"\nApplied ONE-HOT encoding (drop_first=True) to: {nominal_cols}")
print("Reason: these are category labels with no inherent ranking (e.g., 'Dairy' is not")
print("greater or less than 'Snack Foods'). Label-encoding them as 0,1,2... would invent a")
print("false ordinal relationship that the model would wrongly treat as meaningful magnitude.")

X = pd.get_dummies(X, columns=nominal_cols, drop_first=True)

print(f"\nFeature matrix X shape (after encoding): {X.shape}")

#Leak-free train-test split and scaling


X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")
scaler = StandardScaler()
scaler.fit(X_train)  # fit ONLY on training data
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaler fit on X_train only, then used to transform both X_train and X_test.")
print("(Fitting on the full dataset would leak test-set mean/variance into training —")

feature_names = X.columns.tolist()

results={}

Cleaned data has been loaded Successfully with shape:(8523, 13)
Categorical Columns identified are:['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Size', 'Outlet_Location_Type', 'Outlet_Type']

Applied ONE-HOT encoding (drop_first=True) to: ['Item_Fat_Content', 'Item_Type', 'Outlet_Identifier', 'Outlet_Type']
Reason: these are category labels with no inherent ranking (e.g., 'Dairy' is not
greater or less than 'Snack Foods'). Label-encoding them as 0,1,2... would invent a
false ordinal relationship that the model would wrongly treat as meaningful magnitude.

Feature matrix X shape (after encoding): (8523, 35)
X_train: (6818, 35), X_test: (1705, 35)
Scaler fit on X_train only, then used to transform both X_train and X_test.
(Fitting on the full dataset would leak test-set mean/variance into training —


In [2]:
#Decision Tree baseline
dt_full = DecisionTreeClassifier(random_state=42)
dt_full.fit(X_train_scaled, y_clf_train)

train_acc_full = accuracy_score(y_clf_train, dt_full.predict(X_train_scaled))
test_acc_full = accuracy_score(y_clf_test, dt_full.predict(X_test_scaled))

print(f"Unconstrained Decision Tree -> Train accuracy: {train_acc_full:.4f}, Test accuracy: {test_acc_full:.4f}")
print(f"Train-test gap: {train_acc_full - test_acc_full:.4f}")

results["dt_unconstrained"] = {"train_acc": train_acc_full, "test_acc": test_acc_full}

Unconstrained Decision Tree -> Train accuracy: 1.0000, Test accuracy: 0.7408
Train-test gap: 0.2592


In [3]:
#Controlled DecisionTree

dt_controlled = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)
dt_controlled.fit(X_train_scaled, y_clf_train)

train_acc_ctrl = accuracy_score(y_clf_train, dt_controlled.predict(X_train_scaled))
test_acc_ctrl = accuracy_score(y_clf_test, dt_controlled.predict(X_test_scaled))

print(f"Controlled Decision Tree -> Train accuracy: {train_acc_ctrl:.4f}, Test accuracy: {test_acc_ctrl:.4f}")
print(f"Train-test gap: {train_acc_ctrl - test_acc_ctrl:.4f}")

results["dt_controlled"] = {"train_acc": train_acc_ctrl, "test_acc": test_acc_ctrl}

Controlled Decision Tree -> Train accuracy: 0.7985, Test accuracy: 0.7818
Train-test gap: 0.0167


In [4]:
#Gini vs Entropy comparison:

dt_gini = DecisionTreeClassifier(max_depth=5, criterion="gini", random_state=42)
dt_gini.fit(X_train_scaled, y_clf_train)
test_acc_gini = accuracy_score(y_clf_test, dt_gini.predict(X_test_scaled))

dt_entropy = DecisionTreeClassifier(max_depth=5, criterion="entropy", random_state=42)
dt_entropy.fit(X_train_scaled, y_clf_train)
test_acc_entropy = accuracy_score(y_clf_test, dt_entropy.predict(X_test_scaled))

print(f"Gini (max_depth=5)    -> Test accuracy: {test_acc_gini:.4f}")
print(f"Entropy (max_depth=5) -> Test accuracy: {test_acc_entropy:.4f}")

results["gini_entropy"] = {"gini_test_acc": test_acc_gini, "entropy_test_acc": test_acc_entropy}

Gini (max_depth=5)    -> Test accuracy: 0.7818
Entropy (max_depth=5) -> Test accuracy: 0.8088


In [5]:
#Random Forest:

rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train_scaled, y_clf_train)

rf_train_acc = accuracy_score(y_clf_train, rf.predict(X_train_scaled))
rf_test_acc = accuracy_score(y_clf_test, rf.predict(X_test_scaled))
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]
rf_auc = roc_auc_score(y_clf_test, rf_proba)

print(f"Random Forest -> Train accuracy: {rf_train_acc:.4f}, Test accuracy: {rf_test_acc:.4f}, Test AUC: {rf_auc:.4f}")

importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)
top5_importance = importances.head(5)
print("\nTop 5 features by Random Forest importance:")
print(top5_importance.to_string())

results["random_forest"] = {
    "train_acc": rf_train_acc, "test_acc": rf_test_acc, "test_auc": rf_auc,
    "top5_features": top5_importance.to_dict(),
}

Random Forest -> Train accuracy: 0.8439, Test accuracy: 0.8111, Test AUC: 0.8898

Top 5 features by Random Forest importance:
Item_MRP                         0.487963
Item_Visibility                  0.061932
Outlet_Age                       0.061439
Outlet_Establishment_Year        0.057109
Outlet_Type_Supermarket Type1    0.054384


In [6]:
#Gradient Boosting
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(X_train_scaled, y_clf_train)

gb_train_acc = accuracy_score(y_clf_train, gb.predict(X_train_scaled))
gb_test_acc = accuracy_score(y_clf_test, gb.predict(X_test_scaled))
gb_proba = gb.predict_proba(X_test_scaled)[:, 1]
gb_auc = roc_auc_score(y_clf_test, gb_proba)

print(f"Gradient Boosting -> Train accuracy: {gb_train_acc:.4f}, Test accuracy: {gb_test_acc:.4f}, Test AUC: {gb_auc:.4f}")

results["gradient_boosting"] = {"train_acc": gb_train_acc, "test_acc": gb_test_acc, "test_auc": gb_auc}

Gradient Boosting -> Train accuracy: 0.8319, Test accuracy: 0.8147, Test AUC: 0.8938


In [7]:
#Feature ablation study

bottom5_features = importances.tail(5)
print("5 lowest-importance features (from the Task 4 Random Forest):")
print(bottom5_features.to_string())

bottom5_names = bottom5_features.index.tolist()
keep_mask = [f not in bottom5_names for f in feature_names]

X_train_reduced = X_train_scaled[:, keep_mask]
X_test_reduced = X_test_scaled[:, keep_mask]

rf_reduced = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_reduced.fit(X_train_reduced, y_clf_train)
rf_reduced_proba = rf_reduced.predict_proba(X_test_reduced)[:, 1]
rf_reduced_auc = roc_auc_score(y_clf_test, rf_reduced_proba)

print(f"\nFull model (all {len(feature_names)} features)      -> Test AUC: {rf_auc:.4f}")
print(f"Reduced model ({len(feature_names) - 5} features, 5 dropped) -> Test AUC: {rf_reduced_auc:.4f}")
print(f"AUC change (reduced - full): {rf_reduced_auc - rf_auc:+.4f}")

results["ablation"] = {
    "dropped_features": bottom5_names,
    "full_auc": rf_auc,
    "reduced_auc": rf_reduced_auc,
    "auc_change": rf_reduced_auc - rf_auc,
}

5 lowest-importance features (from the Task 4 Random Forest):
Item_Type_Others            0.001392
Item_Type_Hard Drinks       0.001385
Item_Type_Seafood           0.001376
Outlet_Identifier_OUT049    0.001111
Item_Type_Breakfast         0.001024

Full model (all 35 features)      -> Test AUC: 0.8898
Reduced model (30 features, 5 dropped) -> Test AUC: 0.8910
AUC change (reduced - full): +0.0012


In [8]:
#Cross-validated comparison:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", C=1.0, random_state=42),
    "Decision Tree (max_depth=5)": DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42),
}

cv_results = {}
for name, model in cv_models.items():
    scores = cross_val_score(model, X_train_scaled, y_clf_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    cv_results[name] = {"mean_auc": scores.mean(), "std_auc": scores.std()}
    print(f"{name:32s} -> mean AUC: {scores.mean():.4f}, std AUC: {scores.std():.4f}")

results["cv_results"] = cv_results

Logistic Regression              -> mean AUC: 0.8966, std AUC: 0.0089
Decision Tree (max_depth=5)      -> mean AUC: 0.8720, std AUC: 0.0215
Random Forest                    -> mean AUC: 0.8895, std AUC: 0.0087
Gradient Boosting                -> mean AUC: 0.8964, std AUC: 0.0092


In [9]:
#Hyperparameter tuning with GridSearchCV

pipeline = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler(),
    RandomForestClassifier(random_state=42),
)

param_grid = {
    "randomforestclassifier__n_estimators": [50, 100, 200],
    "randomforestclassifier__max_depth": [5, 10, None],
    "randomforestclassifier__min_samples_leaf": [1, 5],
}

n_configs = 1
for v in param_grid.values():
    n_configs *= len(v)
n_folds = 5
print(f"Grid size: {n_configs} hyperparameter configurations x {n_folds} folds = {n_configs * n_folds} total model fits")

grid_search = GridSearchCV(
    pipeline, param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc", n_jobs=-1,
)
# Per the task: fit on X_train and y_clf_train (UNSCALED) -- the pipeline handles scaling.
grid_search.fit(X_train, y_clf_train)

print(f"\nBest params: {grid_search.best_params_}")
print(f"Best CV AUC score: {grid_search.best_score_:.4f}")

best_pipeline = grid_search.best_estimator_
best_pipeline_test_auc = roc_auc_score(y_clf_test, best_pipeline.predict_proba(X_test)[:, 1])
print(f"Best pipeline test-set AUC (on held-out X_test): {best_pipeline_test_auc:.4f}")

results["grid_search"] = {
    "n_total_fits": n_configs * n_folds,
    "best_params": grid_search.best_params_,
    "best_cv_score": grid_search.best_score_,
    "test_auc": best_pipeline_test_auc,
}


Grid size: 18 hyperparameter configurations x 5 folds = 90 total model fits

Best params: {'randomforestclassifier__max_depth': 10, 'randomforestclassifier__min_samples_leaf': 5, 'randomforestclassifier__n_estimators': 100}
Best CV AUC score: 0.8921
Best pipeline test-set AUC (on held-out X_test): 0.8901


In [10]:
#Manual learning curve
fractions = [0.2, 0.4, 0.6, 0.8, 1.0]
learning_curve_rows = []
n_train = len(X_train)

for f in fractions:
    n_rows = int(f * n_train)
    X_sub = X_train.iloc[:n_rows]
    y_sub = y_clf_train.iloc[:n_rows]

    # Fresh clone of the best pipeline's hyperparameters, refit on the subset
    pipeline_f = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        RandomForestClassifier(random_state=42, **{
            k.split("__")[1]: v for k, v in grid_search.best_params_.items()
        }),
    )
    pipeline_f.fit(X_sub, y_sub)

    train_auc_f = roc_auc_score(y_sub, pipeline_f.predict_proba(X_sub)[:, 1])
    test_auc_f = roc_auc_score(y_clf_test, pipeline_f.predict_proba(X_test)[:, 1])

    learning_curve_rows.append({
        "Training fraction": f, "n_rows": n_rows,
        "Training AUC": train_auc_f, "Test AUC": test_auc_f,
    })

learning_curve_df = pd.DataFrame(learning_curve_rows)
print(learning_curve_df.to_string(index=False))

results["learning_curve"] = learning_curve_rows

 Training fraction  n_rows  Training AUC  Test AUC
               0.2    1363      0.944146  0.879818
               0.4    2727      0.934734  0.886382
               0.6    4090      0.929533  0.889757
               0.8    5454      0.927629  0.890183
               1.0    6818      0.925329  0.890112


In [11]:
#Serialize the best model:

joblib.dump(best_pipeline, "best_model.pkl")
print("Saved best pipeline -> best_model.pkl")

# --- Reload-and-predict block (>=5 lines, runs standalone) ---
loaded_model = joblib.load("best_model.pkl")
hand_crafted_rows = X_test.iloc[[0, 1]].copy()   # two real, valid feature rows as stand-ins for "hand-crafted" inputs
reloaded_predictions = loaded_model.predict(hand_crafted_rows)
reloaded_probabilities = loaded_model.predict_proba(hand_crafted_rows)[:, 1]
print(f"Reloaded model predictions on 2 sample rows: {reloaded_predictions}")
print(f"Reloaded model predicted probabilities (class 1): {reloaded_probabilities}")

results["reload_predict"] = {
    "predictions": reloaded_predictions.tolist(),
    "probabilities": reloaded_probabilities.tolist(),
}


Saved best pipeline -> best_model.pkl
Reloaded model predictions on 2 sample rows: [0 0]
Reloaded model predicted probabilities (class 1): [0.19895141 0.19040239]


In [12]:
summary_rows = [
    {"Model": "Logistic Regression (Part 2, C=1.0)",
     "CV Mean AUC": cv_results["Logistic Regression"]["mean_auc"],
     "CV Std AUC": cv_results["Logistic Regression"]["std_auc"],
     "Test AUC": None},  # test AUC for this exact config was reported in Part 2 as 0.8951
    {"Model": "Decision Tree (max_depth=5)",
     "CV Mean AUC": cv_results["Decision Tree (max_depth=5)"]["mean_auc"],
     "CV Std AUC": cv_results["Decision Tree (max_depth=5)"]["std_auc"],
     "Test AUC": roc_auc_score(y_clf_test, dt_controlled.predict_proba(X_test_scaled)[:, 1])},
    {"Model": "Random Forest (n=100, depth=10)",
     "CV Mean AUC": cv_results["Random Forest"]["mean_auc"],
     "CV Std AUC": cv_results["Random Forest"]["std_auc"],
     "Test AUC": rf_auc},
    {"Model": "Gradient Boosting",
     "CV Mean AUC": cv_results["Gradient Boosting"]["mean_auc"],
     "CV Std AUC": cv_results["Gradient Boosting"]["std_auc"],
     "Test AUC": gb_auc},
    {"Model": "Tuned RF Pipeline (GridSearchCV best)",
     "CV Mean AUC": grid_search.best_score_,
     "CV Std AUC": None,
     "Test AUC": best_pipeline_test_auc},
]
summary_df = pd.DataFrame(summary_rows)
# fill Logistic Regression's test AUC from Part 2's reported value for completeness
summary_df.loc[summary_df["Model"].str.contains("Logistic"), "Test AUC"] = 0.8951
print(summary_df.to_string(index=False))

results["summary_table"] = summary_df.to_dict(orient="records")

import json
with open("results_part3.json", "w") as f:
    json.dump(results, f, indent=2, default=str)

print("\nAll Part 3 results saved to results_part3.json")
print("\nDONE.")


                                Model  CV Mean AUC  CV Std AUC  Test AUC
  Logistic Regression (Part 2, C=1.0)     0.896578    0.008935  0.895100
          Decision Tree (max_depth=5)     0.871962    0.021502  0.847986
      Random Forest (n=100, depth=10)     0.889464    0.008715  0.889753
                    Gradient Boosting     0.896384    0.009246  0.893773
Tuned RF Pipeline (GridSearchCV best)     0.892070         NaN  0.890112

All Part 3 results saved to results_part3.json

DONE.
